<a href="https://colab.research.google.com/github/vyenn/ML2024/blob/main/dnn_hw1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Imports

In [1]:
!wget https://github.com/marcin119a/data/raw/refs/heads/main/data_gsn.zip
!unzip data_gsn.zip &> /dev/null
!rm data_gsn.zip

--2025-11-02 23:27:28--  https://github.com/marcin119a/data/raw/refs/heads/main/data_gsn.zip
Resolving github.com (github.com)... 140.82.113.3
Connecting to github.com (github.com)|140.82.113.3|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://raw.githubusercontent.com/marcin119a/data/refs/heads/main/data_gsn.zip [following]
--2025-11-02 23:27:29--  https://raw.githubusercontent.com/marcin119a/data/refs/heads/main/data_gsn.zip
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 5544261 (5.3M) [application/zip]
Saving to: ‘data_gsn.zip’

data_gsn.zip        100%[===================>]   5.29M  --.-KB/s    in 0.05s   

2025-11-02 23:27:30 (103 MB/s) - ‘data_gsn.zip’ saved [5544261/5544261]



In [28]:
import numpy as np
import pandas as pd
import os
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, random_split
from torchvision import transforms
import matplotlib.pyplot as plt
from PIL import Image

## Loading data

In [18]:
data_dir = "data"

transform = transforms.Compose([
    transforms.ToTensor(),              # Convert to tensor with values in [0,1]
    transforms.Normalize((0.5,),(0.5,))  # Linear map [0,1] --> [-1,1]
])

######################
### Loading images ###
######################

image_files = [f for f in os.listdir("data")
               if f.lower().endswith('.png')]

image_tensors = []
for file in image_files:
    img_path = os.path.join("data",file)
    try:
        img = Image.open(img_path).convert("L")  # Ensure grayscale
        tensor = transform(img)
        image_tensors.append(tensor)
    except Exception as e:
        print(f"Skipped {file}: {e}")

# Stack all tensors into one big tensor
if image_tensors:
    images = torch.stack(image_tensors)
    print(f"Loaded {len(images)} images into tensor of shape {images.shape}")
else:
    print("No valid images found in data/")

###################
### Loading csv ###
###################

labels_df = pd.read_csv("data/labels.csv")
columns_of_interest = labels_df.iloc[:, 1:7]
labels = torch.tensor(columns_of_interest.values, dtype=torch.float)
shape_names = columns_of_interest.columns.tolist()

print(f"Loaded {len(labels)} labels into tensor of size {labels.shape}")
print(shape_names)

Loaded 10000 images into tensor of shape torch.Size([10000, 1, 28, 28])
Loaded 10000 labels into tensor of size torch.Size([10000, 6])
['squares', 'circles', 'up', 'right', 'down', 'left']


In [27]:
train_x, val_x = images[:9000], images[9000:]
train_y, val_y = labels[:9000], labels[9000:]

## Dataset class

In [32]:
from torch.utils.data import Dataset

class MyDataset(Dataset):
    def __init__(self, images, cls_labels, cnt_labels):
        # These are your full datasets (e.g., 9000 images, 9000 labels_cls, etc.)
        self.my_images = images         # This is your "train_x"
        self.my_cls_labels = cls_labels # This is your first "train_y"
        self.my_cnt_labels = cnt_labels # This is your second "train_y"

    def __len__(self):
        # Return the total number of samples
        return len(self.my_images)

    def __getitem__(self, idx):
        # Fetch the data at the given index 'idx'
        image = self.my_images[idx]
        cls_label = self.my_cls_labels[idx]
        cnt_label = self.my_cnt_labels[idx]

        # This tuple is what gets passed to the DataLoader
        return (image, cls_label, cnt_label)

# --- How you would use it ---

# 1. Load your raw data (e.g., from numpy arrays or files)
# train_images_data, train_cls_labels_data, train_cnt_labels_data = ...

# 2. Create the dataset object
# train_dataset = MyMultiTaskDataset(train_images_data, train_cls_labels_data, train_cnt_labels_data)

# 3. The DataLoader now works as expected
# train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)

## Creating a model

it should return two outputs (log_probs, counts)

You may add dropout or batch normalization inside the heads, but you must not modify the backbone.



In [22]:
class ConvolutionalNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.feature_extractor = nn.Sequential(
            nn.Conv2d(1, 8, 3, stride=1, padding=1), nn.ReLU(),
            nn.Conv2d(8, 16, 3, stride=1, padding=1), nn.ReLU(),
            nn.Conv2d(16, 32, 3, stride=1, padding=1), nn.ReLU(),
            nn.Conv2d(32, 64, 3, stride=1, padding=1), nn.ReLU(),
            nn.Flatten(start_dim=1),
            nn.Linear(64 * 28 * 28, 256), nn.ReLU()
        )

        # Head 1: Classification
        # Takes the [B, 256] summary and outputs [B, 135] scores
        self.head_cls = nn.Linear(256, 135)
        """
        self.head_cls = nn.Sequential(
            nn.Linear(256, 128),       # An intermediate hidden layer
            nn.ReLU(),
            nn.Dropout(p=0.5),         # Dropout layer for regularization
            nn.Linear(128, 135)        # The final output layer
        )
        """

        # Head 2: Regression
        # Takes the [B, 256] summary and outputs [B, 6] count values
        self.head_cnt = nn.Linear(256, 6)
        """
        self.head_cnt = nn.Sequential(
            nn.Linear(256, 64),
            nn.ReLU(),
            nn.BatchNorm1d(64),         # Add batch normalization
            nn.Linear(64, 6)
        )
        """

        # Helper to convert scores to log-probabilities, as requested
        self.log_softmax = nn.LogSoftmax(dim=1)

    def forward(self, x):
        # x.shape = [B,1,28,28], features.shape = [B,256]
        features = self.feature_extractor(x)

        # Classification head
        cls_logits = self.head_cls(features)
        log_probs = self.log_softmax(cls_logits)

        # Regression head
        counts = self.head_cnt(features)

        return (log_probs, counts)

## Training

In [30]:
torch.manual_seed(1)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Hyperparameters from your requirements
LEARNING_RATE = 1e-3
BATCH_SIZE_TRAIN = 64
BATCH_SIZE_VAL = 1000
N_EPOCHS = 100
TRAIN_SIZE = 9000
VAL_SIZE = 1000

# Early stopping parameters
PATIENCE = 10  # How many epochs to wait after last improvement
best_val_loss = float('inf')
epochs_no_improve = 0
best_model_weights = None

# Ensure dataset is the correct size
if images.shape[0] != TRAIN_SIZE + VAL_SIZE:
    print(f"Warning: Dataset size is {images.shape[0]}, not 10,000.")

# Create DataLoaders
train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE_TRAIN,
    shuffle=True
)
val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE_VAL,
    shuffle=False # No need to shuffle validation data
)


# --- Model, Loss, and Optimizer ---
model = ConvolutionalNet().to(device)
# For head_cls (log-probabilities): Negative Log Likelihood Loss
criterion_cls = nn.NLLLoss()
# For head_cnt (regression): Mean Squared Error Loss (a common choice)
criterion_cnt = nn.MSELoss()

# Optimizer
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)

NameError: name 'train_dataset' is not defined

# TODO

6. in instructions, that is create a Dataset class

In the code cell above, train_dataset and val_dataset should be objects from this class containing x_train, y_train_1 and y_train_2 (respectively with val instead of train)


In [31]:
import copy

# --- The Training Loop ---
print("Starting training...")

for epoch in range(N_EPOCHS):

    model.train()
    running_train_loss = 0.0

    for inputs, labels_cls, labels_cnt in train_loader:
        # Move data to the correct device
        inputs = inputs.to(device)
        labels_cls = labels_cls.to(device)
        labels_cnt = labels_cnt.to(device)

        # Zero the parameter gradients
        optimizer.zero_grad()

        # Forward pass
        log_probs, counts = model(inputs)

        # Calculate losses
        loss_cls = criterion_cls(log_probs, labels_cls)
        # Ensure target is float for regression loss
        loss_cnt = criterion_cnt(counts, labels_cnt.float())

        # Total loss (simple sum)
        # ASSUMPTION: You can weigh these differently, e.g., loss = loss_cls + 0.5 * loss_cnt
        loss = loss_cls + loss_cnt

        # Backward pass and optimize
        loss.backward()
        optimizer.step()

        # Accumulate loss
        running_train_loss += loss.item() * inputs.size(0) # inputs.size(0) == BATCH_SIZE_TRAIN except from last sample since 9000%64=40

    # Calculate average training loss for the epoch
    epoch_train_loss = running_train_loss / len(train_loader.dataset)

    # --- Validation Phase ---
    model.eval()  # Set model to evaluation mode (disables dropout, etc.)
    running_val_loss = 0.0

    with torch.no_grad(): # Disable gradient calculations
        for inputs, labels_cls, labels_cnt in val_loader:
            # Move data to the correct device
            inputs = inputs.to(device)
            labels_cls = labels_cls.to(device)
            labels_cnt = labels_cnt.to(device)

            # Forward pass
            log_probs, counts = model(inputs)

            # Calculate losses
            loss_cls = criterion_cls(log_probs, labels_cls)
            loss_cnt = criterion_cnt(counts, labels_cnt.float())
            loss = loss_cls + loss_cnt

            running_val_loss += loss.item() * inputs.size(0)

    # Calculate average validation loss for the epoch
    epoch_val_loss = running_val_loss / len(val_loader.dataset)

    print(f'Epoch {epoch+1}/{N_EPOCHS} | Train Loss: {epoch_train_loss:.4f} | Val Loss: {epoch_val_loss:.4f}')

    # --- Early Stopping Check ---
    if epoch_val_loss < best_val_loss:
        best_val_loss = epoch_val_loss
        epochs_no_improve = 0
        # Save the weights of the best model
        best_model_weights = copy.deepcopy(model.state_dict())
        print('Validation loss improved. Saving model.')
    else:
        epochs_no_improve += 1

    if epochs_no_improve >= PATIENCE:
        print(f'Early stopping triggered after {epoch+1} epochs.')
        break

print('Training finished.')

Starting training...


ValueError: too many values to unpack (expected 3)